In [ ]:
import json, pandas as pd
from pathlib import Path
from huggingface_hub import HfApi

TOKEN = ""  # insert  HF token here
UPLOAD = Path(".")

# Load annotations
ann = pd.read_csv("Bluesky Moderation Annotations.csv")
ann.columns = ann.columns.str.strip()
ann = ann.drop_duplicates(subset=['annotator_name','post_id'], keep='last')

# Main survey only (display_num < 10000) — both Ines and adash
main_ann = ann[ann['display_num'] < 10000].copy()

# Pilot remaining (display_num 20000-30000)
pilot_rem_ann = ann[(ann['display_num'] >= 20000) & (ann['display_num'] < 30000)].copy()

# Combine both
combined = pd.concat([main_ann, pilot_rem_ann], ignore_index=True)

# Build lookups
lu_ines  = dict(zip(combined[combined['annotator_name']=='Ines']['post_id'],
                    combined[combined['annotator_name']=='Ines']['label']))
lu_adash = dict(zip(combined[combined['annotator_name']=='adash']['post_id'],
                    combined[combined['annotator_name']=='adash']['label']))

# Find posts labeled by both where source is pilot
# Load combined_3k to get source info
main_posts = {}
with open(UPLOAD / "combined_3k.jsonl") as f:
    for line in f:
        if line.strip():
            p = json.loads(line)
            if p.get('uri'): main_posts[p['uri']] = p

# Also load pilot_1k for pilot post metadata
pilot_posts = {}
with open(UPLOAD / "pilot_1k.jsonl") as f:
    for line in f:
        if line.strip():
            p = json.loads(line)
            if p.get('uri'): pilot_posts[p['uri']] = p

# Load pilot_remaining
pilot_rem_posts = {}
with open(UPLOAD / "pilot_remaining.jsonl") as f:
    for line in f:
        if line.strip():
            p = json.loads(line)
            if p.get('uri'): pilot_rem_posts[p['uri']] = p

all_pilot_posts = {**pilot_posts, **pilot_rem_posts}
print(f"Total pilot posts: {len(all_pilot_posts)}")

# Find pilot disagreements (Ines vs adash)
common = set(lu_ines) & set(lu_adash)
pilot_common = [p for p in common if p in all_pilot_posts]
pilot_disagree = [p for p in pilot_common if lu_ines[p] != lu_adash[p]]

print(f"Pilot posts labeled by both: {len(pilot_common)}")
print(f"Pilot disagreements: {len(pilot_disagree)}")

# Get reasons and categories
cats_ines  = dict(zip(combined[combined['annotator_name']=='Ines']['post_id'],
                      combined[combined['annotator_name']=='Ines']['unsafe_categories'].fillna('')))
cats_adash = dict(zip(combined[combined['annotator_name']=='adash']['post_id'],
                      combined[combined['annotator_name']=='adash']['unsafe_categories'].fillna('')))

# Build JSONL
out_rows = []
for uri in pilot_disagree:
    post = dict(all_pilot_posts[uri])
    post['ines_label']       = lu_ines[uri]
    post['adash_label']      = lu_adash[uri]
    post['ines_categories']  = cats_ines.get(uri, '')
    post['adash_categories'] = cats_adash.get(uri, '')
    post['source']           = 'pilot_disagree'
    out_rows.append(post)

print(f"\nBuilding JSONL with {len(out_rows)} posts")
print(f"Ines→Unsafe, adash→Safe: {sum(1 for u in pilot_disagree if lu_ines[u]=='Unsafe')}")
print(f"Ines→Safe, adash→Unsafe: {sum(1 for u in pilot_disagree if lu_adash[u]=='Unsafe')}")

out_path = UPLOAD / "data/pilot_disagreements.jsonl"
with open(out_path, 'w') as f:
    for p in out_rows:
        f.write(json.dumps(p, ensure_ascii=False) + '\n')
print(f"Saved → {out_path}")

# Upload
api = HfApi(token=TOKEN)
api.upload_file(
    path_or_fileobj=str(out_path),
    path_in_repo="pilot_disagreements.jsonl",
    repo_id="ines-abdelaziz/neurips-content-moderation",
    repo_type="dataset",
    token=TOKEN,
)
print("Uploaded to HF!")

Total pilot posts: 1000
Pilot posts labeled by both: 573
Pilot disagreements: 109

Building JSONL with 109 posts
Ines→Unsafe, adash→Safe: 35
Ines→Safe, adash→Unsafe: 74
Saved → data/pilot_disagreements.jsonl
Uploaded to HF!
